# Strata on Kaggle — DevZoneX

Run a Strata-supported model on a Kaggle GPU, save your own private model/runtime caches, and connect OpenCode Desktop through an authenticated temporary Cloudflare Quick Tunnel.

## Before running
1. Enable Internet and an available GPU accelerator in Kaggle notebook settings.
2. Add a Kaggle API token as a Secret named `KAGGLE_API_TOKEN` under Add-ons → Secrets and enable notebook access.
3. Edit the configuration cell with your Kaggle username and a model family/variant supported by Strata.
4. For later sessions, attach your own private model and runtime datasets via Add Input.
5. Read the chosen model's license. Enable the explicit upload confirmation only if the terms permit copying to your own private Kaggle Dataset.

**Compatibility warning:** this is not a universal Hugging Face loader. Strata supports specific families/variants; run and review the launch plan before downloading large files.

## Why this model needs a special setup

The default is **Qwen3.8-Flash-Next Gyro-S**: a **125B total-parameter sparse MoE model**, with around **6B parameters active per token**. It is still a 125B model in terms of total weights/storage, not a 6B model.

- **Gyro-S main model file:** about 58.5 GB; Strata estimates about **27.6 GB combined RAM + VRAM** for the model weights.
- **Typical hardware in our working test:** 2 × NVIDIA Tesla T4 GPUs (~15 GB VRAM each, around 30 GB combined) plus about 31 GB system RAM.
- **The trick:** Strata's Gyro-S quantization compresses the model and manages expert data across VRAM/system RAM, while a large n-gram table stays on disk and is read as needed. That allows a much larger model to run on less GPU VRAM, with a speed/quality trade-off.
- **Cache size:** the complete model dataset is about 79.94 GiB because it includes support/pack files in addition to the main GGUF. Restoring an attached dataset still transfers/copies those files into the fresh Kaggle runtime; it avoids downloading them from the original model host again.

These are estimates, not guaranteed hardware requirements. Strata supports specific model families/variants only. Review the plan and the [Strata model guide](https://github.com/Niko1221/Strata/blob/main/docs/MODELS.md) before launching.

In [ ]:
# ===== USER CONFIGURATION: EDIT THIS CELL =====
KAGGLE_USERNAME = "your-kaggle-username"
MODEL_FAMILY = "gyro"
MODEL_VARIANT = "gyro-s"
CONTEXT = "32k"
VISION = "off"
DRAFTING = "auto"                      # "auto"/"mtp" = model's draft layer (faster); "dflash2" = Qwen3.8-Flash-Next only; "off" = no drafter
GPU_IDS = ""                         # blank = detect all visible NVIDIA GPUs
MODEL_DATA_DIR = "/root/strata-local"

# "copy" = extra local copy, generally faster model loading (default for this tutorial).
# "symlink" = no second 80-GiB local copy, but reading from Kaggle's input mount can be slower.
MODEL_RESTORE_MODE = "copy"

# A public dataset holding the MTP draft layer (mtp/rt), reused so every session
# does not re-download about 5.2 GB. The layer is built from the original BF16
# checkpoint, so it fits every Qwen3.8-Flash-Next variant (Gyro, Unsloth, Q2_0...).
# Only the draft layer is taken from it; weights and packs are never copied from
# here. Leave it blank to let Strata fetch the layer itself.
DRAFT_LAYER_DATASET = "clothespin/strata-qfn-assets-lite"

# Optional: start from a saved Strata preset (keeps tuning, engine settings and
# context between sessions). Flags in this notebook still win for that start.
# Leave STRATA_PRESET blank to ignore presets.
STRATA_PRESET = ""
# Optional: save this cell's choices under this name after a successful launch,
# so a later session can start them with STRATA_PRESET = "<name>".
SAVE_PRESET = ""

import re
_slug = lambda value: re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
MODEL_DATASET_SLUG = _slug(f"strata-model-{MODEL_FAMILY}-{MODEL_VARIANT}")
RUNTIME_DATASET_SLUG = "strata-runtime-cache"

UPLOAD_CACHES = True
I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE = False
FORCE_UPDATE_MODEL_CACHE = False
FORCE_UPDATE_RUNTIME_CACHE = False
RUN_DRAFTING_AB = False               # True = reload the model twice to measure drafting off vs auto (slow)

# GPU-hour guards. The supervision cell stops Strata once nobody is using it;
# returning from that cell lets Kaggle finish the run and release the GPU.
IDLE_MINUTES = 15                     # finish after this many idle minutes; 0 disables the idle check
SERVE_MINUTES = 0                     # hard stop for the supervision cell; 0 = only the idle check

# Quality gate: measure the chat API and check that the model can call tools.
RUN_QUALITY_GATE = False              # several extra minutes; run it once per model
QUALITY_REPEATS = 1

# Optional PEM RSA public key from a local launcher. When set, the API key is
# published encrypted and never appears in this notebook. Leave blank to skip.
CLIENT_PUBLIC_KEY = ""

print("Model selection:", MODEL_FAMILY, MODEL_VARIANT)
print("Model cache:", f"{KAGGLE_USERNAME}/{MODEL_DATASET_SLUG}")
print("Runtime cache:", f"{KAGGLE_USERNAME}/{RUNTIME_DATASET_SLUG}")
print("Drafting:", DRAFTING)
print("Draft layer dataset:", DRAFT_LAYER_DATASET or "(none; Strata downloads it)")
print("Strata preset:", STRATA_PRESET or "(none)")
print("Save preset as:", SAVE_PRESET or "(not saving)")
print("Model private-storage confirmation:", I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE)
print("Model restore mode:", MODEL_RESTORE_MODE)
print("Idle timeout:", f"{IDLE_MINUTES} min" if IDLE_MINUTES else "disabled")
print("Serve limit:", f"{SERVE_MINUTES} min" if SERVE_MINUTES else "none")
print("Quality gate:", RUN_QUALITY_GATE)

## 1. Load helper scripts and Kaggle credentials

This clones the public repository and reads the Kaggle token from Secrets. The helper stores the token in a temporary private config path and never prints it.

In [ ]:
import os, sys, subprocess, json, re, shutil
from pathlib import Path

# This notebook clones the fork that contains the helper code it calls. Pointing
# this at a different repository loads older helpers and fails later with an
# AttributeError, so the checkout is verified before it is used.
REPO_URL = "https://github.com/vinodhkumark/devzonex-strata-kaggle.git"
REPO_DIR = Path("/kaggle/working/devzonex-strata-kaggle")
HELPER = REPO_DIR / "scripts" / "strata_kaggle.py"

# Functions this notebook calls. If any is missing, the checkout is stale.
REQUIRED_HELPERS = (
    "validate_config", "environment_report", "announce_session",
    "serve_until_idle", "run_quality_gate", "publish_connection",
)


def helper_is_current() -> bool:
    try:
        text = HELPER.read_text(encoding="utf-8")
    except OSError:
        return False
    return all(("def " + name) in text for name in REQUIRED_HELPERS)


def sync_repo() -> None:
    """Use the checkout when it is current, otherwise replace it."""
    if REPO_DIR.exists() and helper_is_current():
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
        pull = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
        if pull.returncode == 0 and helper_is_current():
            return
        print("Cached checkout is behind or modified; replacing it from", REPO_URL)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)


sync_repo()

sys.path.insert(0, str(REPO_DIR))
import importlib
from scripts import strata_kaggle as sk
sk = importlib.reload(sk)

MISSING = [name for name in REQUIRED_HELPERS if not hasattr(sk, name)]
if MISSING:
    raise RuntimeError(
        "The helper checkout is missing: " + ", ".join(MISSING) + ".\n"
        "REPO_URL is " + REPO_URL + ".\n"
        "Set REPO_URL to the repository that contains these helpers, then rerun this cell."
    )

KAGGLE_ENV = sk.kaggle_env(KAGGLE_USERNAME, secret_name="KAGGLE_API_TOKEN")
sk.install_kaggle_cli(KAGGLE_ENV)
HARDWARE = sk.hardware_summary()
print(json.dumps(HARDWARE, indent=2))
if not HARDWARE["gpu_names"]:
    raise RuntimeError("No NVIDIA GPU detected. Enable a GPU accelerator in Kaggle settings.")

# Fail before a long launch spends GPU time on a configuration mistake.
CONFIG = sk.validate_config(
    MODEL_FAMILY, MODEL_VARIANT, CONTEXT, DRAFTING, VISION, MODEL_RESTORE_MODE, GPU_IDS,
)
ENVIRONMENT = sk.environment_report()
SESSION = sk.announce_session({
    "family": CONFIG["family"], "variant": CONFIG["variant"],
    "context_tokens": CONFIG["context_tokens"], "drafting": CONFIG["drafting"],
})
print("Kernel session:", SESSION["session_id"] or "(not a Kaggle session)")

## 2. Restore your caches or install Strata

First-time run: no caches need to be attached; Strata can fetch the selected model/engine after the plan is approved.

Returning run: use Add Input to attach your own model-cache and runtime-cache datasets. Kaggle stages the attached dataset into the session. The notebook defaults to MODEL_RESTORE_MODE='copy', which creates a local copy and may improve model reads at the cost of extra restore time and disk space. Set it to 'symlink' to avoid the second copy, but reads from the input mount may be slower.
Runtime restoration safely rewrites absolute symlinks such as `terminal/current` only when their targets stay inside the cached Strata runtime tree. Links that escape the allowed runtime paths are rejected. If an already-open notebook still reports `Unsafe absolute link in runtime archive`, rerun the helper-loading cell to pull the latest fix from GitHub.


In copy mode, the helper shows live byte-level progress, percentage, completed files, copy speed, and elapsed time.


In [ ]:
MODEL_ROOT = Path(MODEL_DATA_DIR)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

RUNTIME_CACHE_RESTORED = sk.restore_runtime_cache(KAGGLE_USERNAME, RUNTIME_DATASET_SLUG)
MODEL_CACHE_RESTORED = sk.restore_model_cache(KAGGLE_USERNAME, MODEL_DATASET_SLUG, MODEL_ROOT, workers=8, family=MODEL_FAMILY, variant=MODEL_VARIANT, restore_mode=MODEL_RESTORE_MODE)

if DRAFT_LAYER_DATASET.strip():
    _draft_owner, _, _draft_slug = DRAFT_LAYER_DATASET.partition("/")
    DRAFT_LAYER_RESTORED = sk.restore_draft_layer(
        _draft_owner.strip(), _draft_slug.strip(), MODEL_ROOT,
        restore_mode=MODEL_RESTORE_MODE,
    )
else:
    DRAFT_LAYER_RESTORED = False
    print("Draft-layer dataset not configured; Strata will fetch it if drafting needs it.")

sk.install_strata_if_missing()

DRAFTING_STATUS = sk.drafting_status(MODEL_ROOT, DRAFTING)

print("Runtime cache restored:", RUNTIME_CACHE_RESTORED)
print("Model cache restored:", MODEL_CACHE_RESTORED)
print("Draft layer restored:", DRAFT_LAYER_RESTORED)
print("Model restore mode:", MODEL_RESTORE_MODE)
print("Strata version check complete.")
print("Drafting:", DRAFTING_STATUS)

## 3. Review the Strata plan

Read the plan output carefully before proceeding. The notebook defaults to `MODEL_RESTORE_MODE = "copy"` for this tutorial: it uses extra disk space and restore time but may provide faster local reads. Set it to `"symlink"` to avoid the second local copy, noting that input-mount reads may be slower and Kaggle still stages the attached dataset into the session. If Strata rejects the selected family/variant, update the configuration cell and rerun this cell. Do not repeatedly run an unsupported plan.

In [ ]:
if not GPU_IDS.strip():
    GPU_IDS = sk.default_gpu_list()
print("GPU selection:", GPU_IDS)
PLAN_OUTPUT = sk.run_plan(
    family=MODEL_FAMILY,
    variant=MODEL_VARIANT,
    context=CONTEXT,
    vision=VISION,
    gpu_list=GPU_IDS,
    data_dir=MODEL_ROOT,
    drafting=DRAFTING,
    preset=STRATA_PRESET or None,
)

## 3b. Optional: measure the drafter (A/B)

Leave `RUN_DRAFTING_AB = False` unless you want a measured comparison. When enabled, this cell reloads the model twice (once with `DRAFTING='off'`, once with `'auto'`), sends the same fixed prompt, and prints tokens/s plus the percentage change. It uses no tunnel and stops the server at the end, so the next cell starts clean. Expect this to take a long time: each run reloads the model, and the first `auto` run also downloads the ~5 GB draft layer.

In [ ]:
if RUN_DRAFTING_AB:
    DRAFTING_AB = sk.compare_drafting(
        family=MODEL_FAMILY, variant=MODEL_VARIANT, context=CONTEXT,
        vision=VISION, gpu_list=GPU_IDS, data_dir=MODEL_ROOT,
        modes=("off", "auto"),
    )
    print(json.dumps(DRAFTING_AB, indent=2))
    print("Set DRAFTING to the faster mode, then rerun the launch cell.")
else:
    print("Drafting A/B disabled. Set RUN_DRAFTING_AB = True in the configuration cell to measure it.")

## 3c. Optional: quality gate

Set `RUN_QUALITY_GATE = True` to measure the chat API and check tool calling
before you connect a client. It reports prompt and decode tokens/s plus
time-to-first-token at three prompt sizes, then asks the model to read, edit
and test a local file through function calls.

The tool check matters more than the speed numbers: a model can decode quickly
and still emit invalid tool calls or edit a file before reading it, which is
what breaks an agent session. This is also how to compare two variants on more
than tokens/s. It adds several minutes to the session.

In [ ]:
if RUN_QUALITY_GATE:
    QUALITY = sk.run_quality_gate(repeats=QUALITY_REPEATS)
else:
    print("Quality gate skipped. Set RUN_QUALITY_GATE = True in the configuration cell to run it.")

## 4. Launch the authenticated Strata server and Cloudflare tunnel

The helper starts Strata with `--access cloudflare`, waits for a loaded model and API-key protection, then starts a Quick Tunnel if Strata did not create one. Quick Tunnel URLs are temporary and publicly reachable; the Strata API key must remain private.

In [ ]:
import requests

# Reuse an already-running server only if it is loaded and API-key protected.
server_is_ready = False
health_json = {}
try:
    r = requests.get("http://127.0.0.1:8080/health", timeout=5)
    health_json = r.json() if r.ok else {}
    server_is_ready = bool(health_json.get("loaded") and health_json.get("api_key"))
except Exception:
    pass

if server_is_ready:
    print("Reusing the healthy, API-key-protected Strata server.")
    if STRATA_PRESET or SAVE_PRESET:
        print("Note: a server is already running, so STRATA_PRESET/SAVE_PRESET are not applied.")
        print("To start the preset instead, run sk.stop_strata() and rerun this cell.")
    LAUNCH_OUTPUT = ""
else:
    if health_json.get("loaded") and not health_json.get("api_key"):
        sk.run_command(["strata", "launch", "--stop"], timeout=60)
    LAUNCH_OUTPUT = sk.launch_strata(
        family=MODEL_FAMILY, variant=MODEL_VARIANT, context=CONTEXT,
        vision=VISION, gpu_list=GPU_IDS, data_dir=MODEL_ROOT,
        drafting=DRAFTING,
        preset=STRATA_PRESET or None,
        save_preset=SAVE_PRESET or None,
    )

match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", LAUNCH_OUTPUT)
LAUNCH_TUNNEL_URL = match.group(0) if match else None
TUNNEL_URL = sk.start_quick_tunnel(LAUNCH_TUNNEL_URL)
HEALTH = sk.wait_health(timeout_seconds=900)
print("Loaded:", HEALTH.get("loaded"))
print("Model:", HEALTH.get("model"))
print("Temporary HTTPS endpoint:", TUNNEL_URL)

# Publish one machine-readable connection record for a local launcher. With
# CLIENT_PUBLIC_KEY set, the API key is encrypted for that client only.
CONNECTION = sk.publish_connection(TUNNEL_URL, client_public_key=CLIENT_PUBLIC_KEY or None)

## 5. Verify the public API and run a short generation

The next cell checks that unauthenticated API requests are rejected with HTTP 401/403, authenticated requests can list models, and a short authenticated chat completion succeeds. It never prints the Strata API key.

In [ ]:
TEST_RESULT = sk.verify_public_endpoint(TUNNEL_URL)
MODEL_ID = TEST_RESULT["model_id"]
print("End-to-end authenticated API test: PASS")
print("Model ID for OpenCode:", MODEL_ID)

## 6. Save your personal private caches

The model and runtime are stored in separate datasets owned by your Kaggle account. Dataset upload is disabled for the model unless you explicitly confirm the model's license permits your private copy. The uploader never passes a public-visibility flag. Kaggle may need time to process datasets after the upload finishes.

In [ ]:
if UPLOAD_CACHES:
    MODEL_UPLOADED = sk.upload_model_cache(
        data_dir=MODEL_ROOT,
        username=KAGGLE_USERNAME,
        slug=MODEL_DATASET_SLUG,
        family=MODEL_FAMILY,
        variant=MODEL_VARIANT,
        env=KAGGLE_ENV,
        license_confirmed=I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE,
        force=FORCE_UPDATE_MODEL_CACHE,
        require_mtp=(DRAFTING in {"auto", "mtp"}),
    )
    RUNTIME_UPLOADED = sk.upload_runtime_cache(
        username=KAGGLE_USERNAME,
        slug=RUNTIME_DATASET_SLUG,
        env=KAGGLE_ENV,
        force=FORCE_UPDATE_RUNTIME_CACHE,
    )
    print("Model cache upload submitted:", MODEL_UPLOADED)
    print("Runtime cache upload submitted:", RUNTIME_UPLOADED)
    print("Check the Kaggle dataset pages and wait for processing to finish before ending this session.")
else:
    print("Cache uploads disabled by configuration.")

## 7. Connect OpenCode Desktop

Merge the printed provider entry into your existing OpenCode configuration; do not replace unrelated providers. Retrieve the Strata API key privately in the Kaggle shell with `strata launch --show-key`. Save it locally on your Windows PC at `~/.config/opencode/strata-key.txt`. Never commit or share the key.

Keep this Kaggle session and tunnel running while using the model. The URL may change after a restart.

In [ ]:
print("OpenCode baseURL:", TUNNEL_URL.rstrip("/") + "/v1")
print("OpenCode model ID:", MODEL_ID)
print("\nMerge this provider entry inside the existing top-level providers object:")
print(sk.opencode_config_snippet(
    TUNNEL_URL, MODEL_ID,
    context_tokens=CONFIG["context_tokens"],
))
print("\nRetrieve the API key privately in the Kaggle terminal; do not copy it to notebook output:")
print("strata launch --show-key")

## 8. Supervise Strata and release the GPU when idle

Run this cell after the API test succeeds, and leave it as the last cell. It
checks the tunnel's authentication boundary once, then watches the server's
request activity. When nobody has used the API for `IDLE_MINUTES`, or
`SERVE_MINUTES` has passed, it stops Strata and returns — a finished notebook
releases the GPU allocation instead of idling until Kaggle's hard limit.

It does not bypass Kaggle's maximum runtime or guarantee uptime; Kaggle can
still stop the session, and the tunnel can still fail.

In [ ]:
import time

# Keep this cell last: returning from it is what lets Kaggle finish the run.
try:
    public_status = requests.get(TUNNEL_URL.rstrip("/") + "/v1/models", timeout=15).status_code
    print("Public endpoint without an API key:", public_status)
    if public_status not in (401, 403):
        print("WARNING: the public endpoint did not reject an unauthenticated request.")
except Exception as exc:
    print("Tunnel check failed:", str(exc)[:180])

SHUTDOWN = sk.serve_until_idle(
    idle_seconds=IDLE_MINUTES * 60,
    serve_seconds=SERVE_MINUTES * 60,
)
print("Session stopped:", SHUTDOWN["reason"], "after", SHUTDOWN["elapsed_seconds"], "seconds.")
print("You can now close this Kaggle session; the GPU allocation is released when the run finishes.")

> **Important:** A running monitor cell is a legitimate health monitor, not an uptime guarantee. Kaggle may end a session at its hard limit, when quotas are exhausted, or if the runtime/server fails. For a reliable always-on endpoint, use infrastructure designed for persistent services.